# 3 · Other catalogs and reusable exports

Copernicus Data Space, Earth Search, and Planetary Computer are included as public STAC API presets. Copernicus is the second option in the map catalog menu. To run this example against Copernicus, change `PRESETS['earthsearch']` to `PRESETS['copernicus']` in the connection cell below; the same Sentinel-2 L2A collection ID and bounded query work with either provider. The saved filename prefix is only a local label. Copernicus uses `https://stac.dataspace.copernicus.eu/v1`; see its [STAC browser](https://browser.stac.dataspace.copernicus.eu/) and [API documentation](https://documentation.dataspace.copernicus.eu/APIs/STAC.html). A custom root must use HTTPS, return a STAC Catalog or Collection, and allow browser CORS. Public metadata access does not guarantee unrestricted asset access. Copernicus Sentinel-2 downloads require provider authentication; this helper does not supply credentials. Some Planetary Computer assets require separate signing. This notebook does not fetch raster assets.

In [ ]:
from stac_browser import STACBrowser, PRESETS
client = await STACBrowser(PRESETS['earthsearch']).connect()
print('Catalog:', client.root.get('title', client.root['id']))
print('Collections:', [c['id'] for c in client.collections])

In [ ]:
items = await client.search(bbox=[-123.1, 37.2, -121.8, 38.1], collection='sentinel-2-l2a', start='2024-06-01', end='2024-06-15', limit=3)
print(len(items), 'items loaded')
if items:
    client.save('earthsearch')
    print('Saved earthsearch-items.geojson and earthsearch-search.json')

## Digital Earth Africa (optional)

**Digital Earth Africa** is the third featured map preset, using the documented `https://explorer.digitalearth.africa/stac/`. The [official documentation](https://docs.digitalearthafrica.org/en/latest/index.html), [direct access guide](https://docs.digitalearthafrica.org/en/latest/platform_tools/direct_access.html), and [STAC download tutorial](https://docs.digitalearthafrica.org/en/latest/sandbox/notebooks/Frequently_used_code/Downloading_data_with_STAC.html) describe its data access.

For a bounded metadata search, uncomment the code below: it selects `s2_l2a`, a small Madagascar rectangle, January 2020, and at most three items. It does not download assets. This example stays optional: on 2026-10-09, normal browser requests failed CORS/network checks and the collections service also returned a non-JSON internal-error page in an independent request. If unavailable, keep using another preset or retry later; this browser app does not bypass CORS or provider failures.

In [ ]:
# Optional: uncomment when Digital Earth Africa is available in your browser.
# africa = await STACBrowser(PRESETS['deafrica']).connect()
# africa_items = await africa.search(
#     collection='s2_l2a', bbox=[45, -20.1, 47, -19.8],
#     start='2020-01-01', end='2020-01-31', limit=3)
# print(len(africa_items), 'Digital Earth Africa items loaded')

## Discover the public STAC Index directory

The map's **STAC Index · Public directory** option lists public APIs and static catalogs alphabetically. Filter by title or URL, inspect the type and source link, then select **Connect to catalog**. Catalog listings are entry points; collections are discovered only after connecting. Filtering and selection make no provider requests. WorldPop remains first and Copernicus second.

The code below reads the same bundled snapshot included with the Explorer. It retains STAC Index source and retrieval metadata. Change `keyword` to filter titles and endpoints (use an empty string for every public listing). The snapshot includes unsupported HTTP and missing-scheme endpoints so you can inspect their original records; do not repair advertised URLs implicitly. An HTTPS URL must also allow browser CORS. Public metadata does not guarantee unrestricted assets.

Credit: [STAC Index public catalogs](https://stacindex.org/catalogs?access=public), from its [official API](https://stacindex.org/api/catalogs). Use **Refresh from STAC Index** in the map for an explicit live, session-only refresh. A failure keeps the current directory and connection; rebuilding the site never refreshes the snapshot. Maintainers update it with `python scripts/update_catalog_directory.py`.

In [ ]:
import json
from pathlib import Path

directory = json.loads(Path('public-catalogs.json').read_text(encoding='utf-8'))
keyword = 'earth'
public_catalogs = directory['catalogs']
assert all(row['access'] == 'public' for row in public_catalogs)
matches = [row for row in public_catalogs
           if keyword.casefold() in (row['title'] + '\n' + row['url']).casefold()]
print('Directory:', len(public_catalogs), 'public listings; fetched', directory['fetched_at'])
print('Source:', directory['source'], '| API:', directory['api'])
for row in matches:
    print(row['title'], '|', 'API' if row['isApi'] else 'Static catalog', '|', row['url'])
selected = matches[0] if matches else None
if selected is None:
    print('No public listings match. Change keyword or use an empty string to list all endpoints.')
# Review the exact selected URL first; connecting below makes provider requests.
# if selected is not None:
#     directory_client = await STACBrowser(selected['url']).connect()
#     print(directory_client.collections)
# Static catalogs: inspect directory_client.browse_links(), then open one chosen link.


## Reuse exported search JSON

The map and Python helper export `stac_root`, `query`, `spatial_mode`, `study_area`, `requests`, retrieval time, and count. The map stores the study-area snapshot from the completed search; later edits and failed searches do not change it. Bounding-box mode sends `bbox`; exact-polygon mode sends GeoJSON `intersects`; no spatial filter sends neither. Collection, date, and page limits still apply. A rectangle can also be sent as an exact polygon. Geometry is limited to one simple closed 2D WGS84 ring with 3–500 vertices, without holes, self-intersections, or date-line edges. Split date-line areas into separate searches. There is no silent polygon-to-bbox fallback. Upload the downloaded JSON to JupyterLite. Replace `saved` below with `json.loads(Path('stac-search.json').read_text(encoding='utf-8'))` to reuse an exported query. Dates come from the inclusive day interval exported by this project.

Static-item exports have `query: null`: their provenance records the item GET request instead of an API search. The replay helper handles that branch first and returns the loaded item as a one-element list. API exports repeat the first search page; call `more()` on a connected client to request further pages. GeoJSON export copies resolve relative links to absolute URLs while the client retains the original metadata.

In [ ]:
import json
from pathlib import Path
saved = client.provenance()
print('Source root:', saved['stac_root'])
print('Reproducible query:', saved.get('query'))

async def replay_saved(saved):
    replay = await STACBrowser(saved['stac_root']).connect()
    query = saved.get('query')
    if query is None:
        requests = saved.get('requests') or []
        if not requests or requests[-1].get('method', 'GET').upper() != 'GET':
            raise ValueError('Static-item replay requires an exported item GET request.')
        await replay.open_link({'rel': 'item', 'href': requests[-1]['url']})
        return replay.items
    dates = query.get('datetime', '../..').split('/')
    return await replay.search(
        bbox=query.get('bbox'), intersects=query.get('intersects'),
        study_area=saved.get('study_area'), collection=(query.get('collections') or [None])[0],
        start=None if dates[0] == '..' else dates[0][:10],
        end=None if dates[1] == '..' else dates[1][:10], limit=query['limit'])

# To issue the saved request again (new HTTP requests), uncomment:
# replay_items = await replay_saved(saved)

## Static catalogs are different

A static Catalog/Collection may have `child` and `item` links without a remote `search` endpoint. Connect with `await STACBrowser('https://.../catalog.json').connect()`, inspect `client.browse_links()`, and pass a selected link to `await client.open_link(link)`. Child links return another connected browser; item links load one item for export. This does not pretend to provide server-side search or recursively crawl the catalog.

## Troubleshooting

A request failure can mean CORS, offline networking, an invalid TLS certificate, a provider error, or non-JSON content. There is no CORS bypass. Correct the URL, choose another preset, or retry later. Dates, bbox order, polygon geometry, and page sizes are validated before requests. To query an exact area directly, call `await client.search(intersects={"type": "Polygon", "coordinates": [[[-123.1, 37.2], [-121.8, 37.2], [-122.4, 38.1], [-123.1, 37.2]]]}, collection='sentinel-2-l2a', start='2024-06-01', end='2024-06-15', limit=3)`. To omit the study-area spatial filter, call `await client.search(collection='sentinel-2-l2a', start='2024-06-01', end='2024-06-15', limit=3)`. Do not pass both `bbox` and `intersects`. The map’s **Field guide** link includes architecture and references.